> 遇到不可微的随机节点（采样），即梯度无法通过”采样“传播。在标准的计算图中，梯度绝对无法直接穿透“采样（Sampling）”这个随机操作（Stochastic Operation）进行反向传播。通常有两条截然不同的处理路线。

> $a \sim \pi_\theta(a|s)$

$$
\nabla_\theta \mathbb{E}_{a \sim \pi_\theta}[R(a)] = \mathbb{E}_{a \sim \pi_\theta}[R(a) \nabla_\theta \log \pi_\theta(a)]
$$

- 黑盒路线（Score Function Trick）：Policy Gradient，它将环境视为黑盒，完全通过“试错”来调整输出概率。
    - $\nabla_\theta \log p_\theta(\tau)=\frac{\nabla_\theta p_\theta(\tau)}{p_\theta(\tau)}$，概率的相对变化率，也叫 score。
        - 在统计学中就是标准的“得分函数（Score Function）”。它的数学期望恒为 0，
    - 优势：极度通用。无论是大语言模型生成离散的 Token，还是机器人输出连续的扭矩，只要能采样、能算概率，就能用。
    - 劣势：方差极大。因为它相当于蒙上眼睛在广袤的空间里瞎子摸象，极容易受到极值样本的干扰。
- 类比：就像是一个射击运动员（参数 $\theta$）打出了子弹（采样）。子弹飞在空中受风速等随机影响（黑盒环境与采样随机性），轨迹无法求导计算。教练（优化器）不要求计算子弹轨迹的导数，而是只看最终上靶的环数（$R$），然后直接指导运动员：“下次你的枪口（概率分布的中心）往左偏一点”。

> $a \sim \mathcal{N}(\mu_\theta, \sigma_\theta^2)$

$$
a = \mu_\theta + \sigma_\theta \odot \epsilon \quad \text{其中} \quad \epsilon \sim \mathcal{N}(0, 1)
$$
- 动作 $a$ 变成了一个关于参数 $\theta$ 的确定性可微函数。唯一的随机性被“隔离”在了标准高斯噪声 $\epsilon$ 中。
- 白盒路线（Reparameterization Trick，重参数化技巧）：VAE
    - 它的核心思想是把随机性作为一个纯粹的外部噪声（如高斯噪声 $\epsilon$）剥离出来，让梯度直接“穿透”模型。
- 类比：如果 Score Function 是“考完试只看总分，然后盲目调整接下来的学习状态”，那么重参数化就是“直接拿着标准答案去一步步纠正解题步骤”。后者方差极小，但苛刻的前提是这个“过程”必须在数学上全程可微，无法直接应用于离散动作（如文本生成）。